# 01. Asynchronous API Ingestion & Concurrency
**FDE Preparation Series**

This notebook demonstrates production-grade async data fetching using `httpx` and `asyncio`, including rate limiting, retry handling, and nest-asyncio for Google Colab compatibility.

In [ ]:
# Install dependencies in Google Colab
!pip install -q httpx pydantic nest-asyncio

In [ ]:
import asyncio
import nest_asyncio
import httpx
from typing import List, Dict, Any

# Enable nested event loops in Colab environment
nest_asyncio.apply()

In [ ]:
async def fetch_item(client: httpx.AsyncClient, item_id: int, semaphore: asyncio.Semaphore) -> Dict[str, Any]:
    url = f"https://jsonplaceholder.typicode.com/posts/{item_id}"
    async with semaphore:
        try:
            resp = await client.get(url, timeout=10.0)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f"[Error] Item {item_id}: {e}")
            return {}

async def main():
    semaphore = asyncio.Semaphore(5) # max 5 parallel requests
    async with httpx.AsyncClient() as client:
        tasks = [fetch_item(client, i, semaphore) for i in range(1, 11)]
        results = await asyncio.gather(*tasks)
        print(f"Fetched {len(results)} items successfully!")
        print("Sample item 1:", results[0])

# Run in Colab event loop
asyncio.run(main())